# Procesamiento y Consolidacion de Datos - TFM Extorsion Colombia

Este notebook realiza:
1. Carga de datos desde APIs y archivos
2. Limpieza y estandarizacion de cada dataset
3. Exportacion de datasets limpios individuales
4. Merge consolidado por ano, mes, codigo de departamento y codigo de municipio
5. Exportacion del consolidado final

**Nota**: La agregacion se realiza a nivel municipio-ano-mes para los datasets con fecha detallada.

## 1. Configuracion Inicial

In [1]:
import pandas as pd
import numpy as np
import requests
import os
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# Directorios - Arquitectura Medallion
BASE_PATH = 'generated_data'
SILVER_PATH = f'{BASE_PATH}/silver'  # Limpios SIN agregar
GOLD_PATH = f'{BASE_PATH}/gold'      # Agregados para modelación

os.makedirs(SILVER_PATH, exist_ok=True)
os.makedirs(GOLD_PATH, exist_ok=True)

print("Librerias cargadas y directorios creados.")
print(f"SILVER: {SILVER_PATH}/")
print(f"GOLD:   {GOLD_PATH}/")

Librerias cargadas y directorios creados.
SILVER: generated_data/silver/
GOLD:   generated_data/gold/


In [2]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def fetch_api_data(api_url: str, dataset_name: str, batch_size: int = 50000) -> pd.DataFrame:
    """
    Obtiene datos de API Socrata con paginacion automatica.
    Maneja datasets grandes (>1000 registros).
    """
    print(f"Cargando '{dataset_name}'...")
    
    all_data = []
    offset = 0
    
    try:
        while True:
            url = f"{api_url}?$limit={batch_size}&$offset={offset}&$order=:id"
            response = requests.get(url, timeout=120)
            response.raise_for_status()
            batch = response.json()
            
            if not batch or not isinstance(batch, list):
                break
            
            all_data.extend(batch)
            print(f"  ... {len(all_data):,} registros")
            
            if len(batch) < batch_size:
                break
                
            offset += batch_size
        
        if not all_data:
            print(f"  [AVISO] Sin datos")
            return pd.DataFrame()
        
        df = pd.DataFrame(all_data)
        print(f"  [OK] Total: {df.shape[0]:,} filas x {df.shape[1]} columnas")
        return df
    
    except Exception as e:
        print(f"  [ERROR] {e}")
        return pd.DataFrame()


def parse_date_column(series: pd.Series, date_format: str = None) -> pd.Series:
    """
    Parsea columna de fechas de manera robusta.
    Maneja formatos ISO (2003-01-01T00:00:00.000) y otros formatos comunes.
    
    Args:
        series: Serie de pandas con fechas
        date_format: Formato especifico (opcional). Si es None, se infiere.
    
    Returns:
        Serie de tipo datetime
    """
    if date_format:
        return pd.to_datetime(series, format=date_format, errors='coerce')
    
    # Intentar formato ISO primero (comun en APIs)
    result = pd.to_datetime(series, format='%Y-%m-%dT%H:%M:%S.%f', errors='coerce')
    
    # Si hay muchos NaT, intentar otros formatos
    if result.isna().sum() > len(result) * 0.5:
        result = pd.to_datetime(series, format='%Y-%m-%d', errors='coerce')
    
    if result.isna().sum() > len(result) * 0.5:
        result = pd.to_datetime(series, format='%d/%m/%Y', errors='coerce')
    
    if result.isna().sum() > len(result) * 0.5:
        result = pd.to_datetime(series, infer_datetime_format=True, errors='coerce')
    
    return result


def extract_year_month(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    Extrae ano y mes de una columna de fecha.
    
    Args:
        df: DataFrame
        date_col: Nombre de la columna con la fecha
    
    Returns:
        DataFrame con columnas 'ano' y 'mes' agregadas
    """
    df = df.copy()
    df[date_col] = parse_date_column(df[date_col])
    df['ano'] = df[date_col].dt.year
    df['mes'] = df[date_col].dt.month
    return df


def standardize_codes(df: pd.DataFrame, cod_dpto_col: str, cod_mun_col: str = None) -> pd.DataFrame:
    """
    Estandariza codigos DANE:
    - cod_dpto: string de 2 digitos con padding (ej: '05')
    - cod_mun: string de 5 digitos con padding (ej: '05001')
    """
    df = df.copy()
    df['cod_dpto'] = df[cod_dpto_col].astype(str).str.strip().str.zfill(2)
    
    if cod_mun_col and cod_mun_col in df.columns:
        df['cod_mun'] = df[cod_mun_col].astype(str).str.strip().str.zfill(5)
    
    return df


def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia nombres de columnas: minusculas, sin espacios, sin acentos.
    """
    df = df.copy()
    replacements = {
        'á': 'a', 'é': 'e', 'í': 'i', 'ó': 'o', 'ú': 'u', 'ñ': 'n',
        'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U', 'Ñ': 'N'
    }
    new_cols = df.columns.str.lower().str.strip().str.replace(' ', '_')
    for old, new in replacements.items():
        new_cols = new_cols.str.replace(old, new)
    df.columns = new_cols
    return df


def export_clean(df: pd.DataFrame, name: str):
    """Exporta dataset limpio a CSV."""
    path = f"{OUTPUT_PATH}/clean/{name}_clean.csv"
    df.to_csv(path, index=False)
    print(f"  Exportado: {path}")


def show_info(df: pd.DataFrame, name: str):
    """Muestra resumen del dataset."""
    print(f"\n{'='*60}")
    print(f"{name.upper()}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas")
    print(f"Columnas: {list(df.columns)}")
    print(f"Tipos:\n{df.dtypes.to_string()}")
    print(f"\nPrimeras filas:")
    display(df.head(3))

def export_silver(df: pd.DataFrame, name: str):
    """Exporta dataset SILVER (limpio sin agregar)."""
    path = f"{SILVER_PATH}/{name}_silver.csv"
    df.to_csv(path, index=False)
    print(f"  [SILVER] {path}")


def export_gold(df: pd.DataFrame, name: str):
    """Exporta dataset GOLD (agregado para modelación)."""
    path = f"{GOLD_PATH}/{name}_gold.csv"
    df.to_csv(path, index=False)
    print(f"  [GOLD]   {path}")

## 2. Carga y Limpieza de Datasets

Cada dataset se limpia con:
- Nombres de columnas estandarizados
- Codigos DANE normalizados (cod_dpto: 2 digitos, cod_mun: 5 digitos)
- Fechas parseadas correctamente (formato ISO: YYYY-MM-DDTHH:MM:SS.fff)
- Ano y mes extraidos como enteros
- Agregacion a nivel municipio-ano-mes

### 2.1 Extorsion (Variable Objetivo)

In [3]:
# Cargar datos
df_extorsion_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/q2ib-t9am.json',
    dataset_name='Extorsion'
)

# Limpieza
df_extorsion = df_extorsion_raw.copy()
df_extorsion = standardize_codes(df_extorsion, 'cod_depto', 'cod_muni')
df_extorsion = extract_year_month(df_extorsion, 'fecha_hecho')
df_extorsion['cantidad'] = pd.to_numeric(df_extorsion['cantidad'], errors='coerce').fillna(0).astype(int)
df_extorsion = df_extorsion.rename(columns={
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre'
})

# SILVER (todos los registros con fecha)
df_extorsion_silver = df_extorsion[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

show_info(df_extorsion_silver, 'Extorsion - SILVER')
export_silver(df_extorsion_silver, 'extorsion')

# GOLD (agregado mensual)
df_extorsion_gold = (df_extorsion_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_extorsion=('cantidad', 'sum'))
    .reset_index()
)

df_extorsion_gold = df_extorsion_gold.astype({
    'cod_dpto': str,
    'cod_mun': str,
    'ano': int,
    'mes': int,
    'total_extorsion': int
})

show_info(df_extorsion_gold, 'Extorsion - GOLD')
export_gold(df_extorsion_gold, 'extorsion')

Cargando 'Extorsion'...
  ... 50,000 registros
  ... 100,000 registros
  ... 120,345 registros
  [OK] Total: 120,345 filas x 6 columnas

EXTORSION - SILVER
Shape: 120,345 filas x 8 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'fecha_hecho', 'ano', 'mes', 'cantidad']
Tipos:
cod_dpto               object
dpto_nombre            object
cod_mun                object
mun_nombre             object
fecha_hecho    datetime64[ns]
ano                     int32
mes                     int32
cantidad                int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,fecha_hecho,ano,mes,cantidad
0,50,META,50001,VILLAVICENCIO,2003-01-01,2003,1,1
1,18,CAQUETA,18001,FLORENCIA,2003-01-01,2003,1,1
2,70,SUCRE,70742,SAN LUIS DE SINCE,2003-01-01,2003,1,1


  [SILVER] generated_data/silver/extorsion_silver.csv

EXTORSION - GOLD
Shape: 38,339 filas x 7 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_extorsion']
Tipos:
cod_dpto           object
dpto_nombre        object
cod_mun            object
mun_nombre         object
ano                 int64
mes                 int64
total_extorsion     int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_extorsion
0,05,ANTIOQUIA,05001,MEDELLIN,2003,1,23
1,05,ANTIOQUIA,05001,MEDELLIN,2003,2,15
2,05,ANTIOQUIA,05001,MEDELLIN,2003,3,9


  [GOLD]   generated_data/gold/extorsion_gold.csv


### 2.2 Poblacion

**Nota**: Dataset de poblacion solo tiene informacion anual, no mensual.

In [13]:
# Cargar datos
excel_url = "https://www.dane.gov.co/files/censo2018/proyecciones-de-poblacion/Municipal/DCD-area-proypoblacion-Mun-2020-2035-ActPostCOVID-19.xlsx"
print("Cargando 'Poblacion'...")
df_poblacion_raw = pd.read_excel(excel_url, sheet_name="Hoja1", header=8)
print(f"  [OK] {df_poblacion_raw.shape[0]:,} filas x {df_poblacion_raw.shape[1]} columnas")

Cargando 'Poblacion'...
  [OK] 53,859 filas x 7 columnas


In [14]:
# Limpieza
df_poblacion = df_poblacion_raw.copy()
df_poblacion = df_poblacion.rename(columns={
    'DP': 'cod_dpto',
    'DPNOM': 'dpto_nombre',
    'MPIO': 'cod_mun',
    'DPMP': 'mun_nombre',
    'AÑO': 'ano',
    'Población': 'poblacion',
    'ÁREA GEOGRÁFICA': 'area_geografica'
})

df_poblacion = df_poblacion[df_poblacion['ano'].notna() & df_poblacion['cod_mun'].notna()]

# Mapeo de areas
area_map = {
    'Cabecera Municipal': 'Urbana',
    'Centros Poblados y Rural Disperso': 'Rural',
    'Total': 'Total'
}
df_poblacion = df_poblacion[df_poblacion['area_geografica'].isin(area_map)]
df_poblacion['tipo_area'] = df_poblacion['area_geografica'].map(area_map)

# Estandarizar codigos
df_poblacion['cod_dpto'] = df_poblacion['cod_dpto'].astype(str).str.zfill(2)
df_poblacion['cod_mun'] = df_poblacion['cod_mun'].astype(str).str.split('.').str[0].str.zfill(5)
df_poblacion['ano'] = df_poblacion['ano'].astype(int)
df_poblacion['poblacion'] = pd.to_numeric(df_poblacion['poblacion'], errors='coerce').astype('Int64')

# SILVER (Urbana, Rural, Total separados)
df_poblacion_silver = df_poblacion[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 
    'ano', 'tipo_area', 'poblacion'
]].copy()

show_info(df_poblacion_silver, 'Poblacion - SILVER')
export_silver(df_poblacion_silver, 'poblacion')

# GOLD (pivotado)
df_poblacion_gold = (df_poblacion_silver
    .pivot_table(
        index=['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano'],
        columns='tipo_area',
        values='poblacion',
        aggfunc='first'
    )
    .reset_index()
)
df_poblacion_gold.columns.name = None

df_poblacion_gold = df_poblacion_gold.rename(columns={
    'Rural': 'poblacion_rural',
    'Total': 'poblacion_total',
    'Urbana': 'poblacion_urbana'
})

df_poblacion_gold = df_poblacion_gold.astype({
    'ano': int,
    'poblacion_rural': 'Int64',
    'poblacion_total': 'Int64',
    'poblacion_urbana': 'Int64'
})

show_info(df_poblacion_gold, 'Poblacion - GOLD')
export_gold(df_poblacion_gold, 'poblacion')


POBLACION - SILVER
Shape: 53,856 filas x 7 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'tipo_area', 'poblacion']
Tipos:
cod_dpto       object
dpto_nombre    object
cod_mun        object
mun_nombre     object
ano             int64
tipo_area      object
poblacion       Int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,tipo_area,poblacion
0,05,Antioquia,05001,Medellín,2020,Urbana,2476569
1,05,Antioquia,05001,Medellín,2020,Rural,43023
2,05,Antioquia,05001,Medellín,2020,Total,2519592


  [SILVER] generated_data/silver/poblacion_silver.csv

POBLACION - GOLD
Shape: 17,952 filas x 8 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'poblacion_rural', 'poblacion_total', 'poblacion_urbana']
Tipos:
cod_dpto            object
dpto_nombre         object
cod_mun             object
mun_nombre          object
ano                  int64
poblacion_rural      Int64
poblacion_total      Int64
poblacion_urbana     Int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,poblacion_rural,poblacion_total,poblacion_urbana
0,05,Antioquia,05001,Medellín,2020,43023,2519592,2476569
1,05,Antioquia,05001,Medellín,2021,42352,2549008,2506656
2,05,Antioquia,05001,Medellín,2022,41952,2572350,2530398


  [GOLD]   generated_data/gold/poblacion_gold.csv


### 2.3 Homicidio

In [5]:
# Cargar datos
df_homicidio_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/m8fd-ahd9.json',
    dataset_name='Homicidio'
)

# Limpieza
df_homicidio = df_homicidio_raw.copy()
df_homicidio = standardize_codes(df_homicidio, 'cod_depto', 'cod_muni')
df_homicidio = extract_year_month(df_homicidio, 'fecha_hecho')
df_homicidio['cantidad'] = pd.to_numeric(df_homicidio['cantidad'], errors='coerce').fillna(0).astype(int)
df_homicidio = df_homicidio.rename(columns={
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre'
})

# SILVER
df_homicidio_silver = df_homicidio[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

show_info(df_homicidio_silver, 'Homicidio - SILVER')
export_silver(df_homicidio_silver, 'homicidio')

# GOLD
df_homicidio_gold = (df_homicidio_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_homicidio=('cantidad', 'sum'))
    .reset_index()
)

df_homicidio_gold = df_homicidio_gold.astype({
    'cod_dpto': str,
    'cod_mun': str,
    'ano': int,
    'mes': int,
    'total_homicidio': int
})

show_info(df_homicidio_gold, 'Homicidio - GOLD')
export_gold(df_homicidio_gold, 'homicidio')

Cargando 'Homicidio'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 333,293 registros
  [OK] Total: 333,293 filas x 8 columnas

HOMICIDIO - SILVER
Shape: 333,293 filas x 8 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'fecha_hecho', 'ano', 'mes', 'cantidad']
Tipos:
cod_dpto               object
dpto_nombre            object
cod_mun                object
mun_nombre             object
fecha_hecho    datetime64[ns]
ano                     int32
mes                     int32
cantidad                int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,fecha_hecho,ano,mes,cantidad
0,11,BOGOTA D.C.,11001,BOGOTA D.C.,2003-01-01,2003,1,1
1,11,BOGOTA D.C.,11001,BOGOTA D.C.,2003-01-01,2003,1,1
2,11,BOGOTA D.C.,11001,BOGOTA D.C.,2003-01-01,2003,1,1


  [SILVER] generated_data/silver/homicidio_silver.csv

HOMICIDIO - GOLD
Shape: 88,462 filas x 7 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_homicidio']
Tipos:
cod_dpto           object
dpto_nombre        object
cod_mun            object
mun_nombre         object
ano                 int64
mes                 int64
total_homicidio     int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_homicidio
0,05,ANTIOQUIA,05001,MEDELLIN,2003,1,210
1,05,ANTIOQUIA,05001,MEDELLIN,2003,2,167
2,05,ANTIOQUIA,05001,MEDELLIN,2003,3,199


  [GOLD]   generated_data/gold/homicidio_gold.csv


### 2.4 Hurto

In [6]:
# Cargar datos
df_hurto_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/4rxi-8m8d.json',
    dataset_name='Hurto'
)

# Limpieza
df_hurto = df_hurto_raw.copy()
df_hurto = standardize_codes(df_hurto, 'cod_depto', 'cod_muni')
df_hurto = extract_year_month(df_hurto, 'fecha_hecho')
df_hurto['cantidad'] = pd.to_numeric(df_hurto['cantidad'], errors='coerce').fillna(0).astype(int)
df_hurto = df_hurto.rename(columns={
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre'
})

# SILVER
df_hurto_silver = df_hurto[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

show_info(df_hurto_silver, 'Hurto - SILVER')
export_silver(df_hurto_silver, 'hurto')

# GOLD
df_hurto_gold = (df_hurto_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_hurto=('cantidad', 'sum'))
    .reset_index()
)

df_hurto_gold = df_hurto_gold.astype({
    'cod_dpto': str,
    'cod_mun': str,
    'ano': int,
    'mes': int,
    'total_hurto': int
})

show_info(df_hurto_gold, 'Hurto - GOLD')
export_gold(df_hurto_gold, 'hurto')

Cargando 'Hurto'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 350,000 registros
  ... 400,000 registros
  ... 450,000 registros
  ... 500,000 registros
  ... 550,000 registros
  ... 600,000 registros
  ... 622,955 registros
  [OK] Total: 622,955 filas x 6 columnas

HURTO - SILVER
Shape: 622,955 filas x 8 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'fecha_hecho', 'ano', 'mes', 'cantidad']
Tipos:
cod_dpto               object
dpto_nombre            object
cod_mun                object
mun_nombre             object
fecha_hecho    datetime64[ns]
ano                     int32
mes                     int32
cantidad                int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,fecha_hecho,ano,mes,cantidad
0,73,TOLIMA,73001,IBAGUE,2003-01-01,2003,1,6
1,08,ATLANTICO,08001,BARRANQUILLA,2003-01-01,2003,1,4
2,68,SANTANDER,68276,FLORIDABLANCA,2003-01-01,2003,1,1


  [SILVER] generated_data/silver/hurto_silver.csv

HURTO - GOLD
Shape: 109,660 filas x 7 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_hurto']
Tipos:
cod_dpto       object
dpto_nombre    object
cod_mun        object
mun_nombre     object
ano             int64
mes             int64
total_hurto     int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_hurto
0,05,ANTIOQUIA,05001,MEDELLIN,2003,1,99
1,05,ANTIOQUIA,05001,MEDELLIN,2003,2,87
2,05,ANTIOQUIA,05001,MEDELLIN,2003,3,127


  [GOLD]   generated_data/gold/hurto_gold.csv


### 2.5 Terrorismo

In [7]:
# Cargar datos
df_terrorismo_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/yi5j-5fe9.json',
    dataset_name='Terrorismo'
)

# Limpieza
df_terrorismo = df_terrorismo_raw.copy()
df_terrorismo = standardize_codes(df_terrorismo, 'cod_depto', 'cod_muni')
df_terrorismo = extract_year_month(df_terrorismo, 'fecha_hecho')
df_terrorismo['cantidad'] = pd.to_numeric(df_terrorismo['cantidad'], errors='coerce').fillna(0).astype(int)
df_terrorismo = df_terrorismo.rename(columns={
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre'
})

# SILVER
df_terrorismo_silver = df_terrorismo[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha_hecho', 'ano', 'mes', 'cantidad'
]].copy()

show_info(df_terrorismo_silver, 'Terrorismo - SILVER')
export_silver(df_terrorismo_silver, 'terrorismo')

# GOLD
df_terrorismo_gold = (df_terrorismo_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(total_terrorismo=('cantidad', 'sum'))
    .reset_index()
)

df_terrorismo_gold = df_terrorismo_gold.astype({
    'cod_dpto': str,
    'cod_mun': str,
    'ano': int,
    'mes': int,
    'total_terrorismo': int
})

show_info(df_terrorismo_gold, 'Terrorismo - GOLD')
export_gold(df_terrorismo_gold, 'terrorismo')

Cargando 'Terrorismo'...
  ... 14,596 registros
  [OK] Total: 14,596 filas x 7 columnas

TERRORISMO - SILVER
Shape: 14,596 filas x 8 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'fecha_hecho', 'ano', 'mes', 'cantidad']
Tipos:
cod_dpto               object
dpto_nombre            object
cod_mun                object
mun_nombre             object
fecha_hecho    datetime64[ns]
ano                     int32
mes                     int32
cantidad                int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,fecha_hecho,ano,mes,cantidad
0,52,NARIÑO,52356,IPIALES,2003-01-01,2003,1,1
1,05,ANTIOQUIA,05660,SAN LUIS,2003-01-01,2003,1,1
2,52,NARIÑO,52356,IPIALES,2003-01-01,2003,1,1


  [SILVER] generated_data/silver/terrorismo_silver.csv

TERRORISMO - GOLD
Shape: 9,369 filas x 7 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_terrorismo']
Tipos:
cod_dpto            object
dpto_nombre         object
cod_mun             object
mun_nombre          object
ano                  int64
mes                  int64
total_terrorismo     int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_terrorismo
0,05,ANTIOQUIA,05001,MEDELLIN,2003,2,1
1,05,ANTIOQUIA,05001,MEDELLIN,2003,5,2
2,05,ANTIOQUIA,05001,MEDELLIN,2003,6,1


  [GOLD]   generated_data/gold/terrorismo_gold.csv


### 2.6 Coca (Cultivos Ilicitos)

**Nota**: Dataset solo tiene informacion anual, no mensual.

In [16]:
# Cargar datos
df_coca_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/acs4-3wgp.json',
    dataset_name='Coca'
)

# Limpieza
df_coca = df_coca_raw.copy()

df_coca = df_coca.rename(columns={
    'coddepto': 'cod_dpto',
    'codmpio': 'cod_mun',
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre'
})

# Identificar columnas de anos
year_cols = [col for col in df_coca.columns if col.startswith('_') and col[1:].isdigit()]

Cargando 'Coca'...
  ... 319 registros
  [OK] Total: 319 filas x 27 columnas


In [17]:
df_coca.head()

,cod_dpto,dpto_nombre,cod_mun,mun_nombre,_2001,_2002,_2003,_2004,_2005,_2006,...,_2014,_2015,_2016,_2017,_2018,_2019,_2020,_2021,_2022,_2023
0,91,AMAZONAS,91263,EL ENCANTO (Cor. Departamental),191.8,264.0,164.0,270.0,382.0,233.0,...,20.0,11.8,12.7,8.0,4.1,2.5,0,- 0,- 0,- 0
1,91,AMAZONAS,91405,LA CHORRERA (Cor. Departamental),65.0,236.0,209.0,271.0,257.0,223.0,...,51.0,13.6,14.1,18.2,6.0,3.2,0,- 0,- 0,- 0
2,91,AMAZONAS,91407,LA PEDRERA (Cor. Departamental),- 0,- 0,- 0,- 0,- 0,- 0,...,- 0,- 0,- 0,- 0,- 0,- 0,0,- 0,- 0,- 0
3,91,AMAZONAS,91430,LA VICTORIA (Pacoa) (Cor. Departamental),- 0,- 0,- 0,- 0,- 0,- 0,...,- 0,- 0,- 0,- 0,- 0,- 0,0,- 0,- 0,- 0
4,91,AMAZONAS,91460,MIRITÍ-PARANÁ (Campoamor) (Cor. Departamental),6.0,43.0,36.2,30.0,12.0,4.0,...,- 0,- 0,- 0,- 0,- 0,- 0,0,- 0,- 0,- 0


In [18]:
# Transformar de ancho a largo
df_coca_long = df_coca.melt(
    id_vars=['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre'],
    value_vars=year_cols,
    var_name='ano',
    value_name='hectareas'
)

df_coca_long['ano'] = df_coca_long['ano'].str.replace('_', '').astype(int)

df_coca_long['hectareas'] = (df_coca_long['hectareas']
    .astype(str)
    .str.replace('- 0', '0')
    .str.replace('-', '')
    .str.strip()
)
df_coca_long['hectareas'] = pd.to_numeric(df_coca_long['hectareas'], errors='coerce').fillna(0)

df_coca_long['cod_dpto'] = df_coca_long['cod_dpto'].astype(str).str.zfill(2)
df_coca_long['cod_mun'] = df_coca_long['cod_mun'].astype(str).str.zfill(5)

# SILVER (hectáreas por municipio y año)
df_coca_silver = df_coca_long[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'ano', 'hectareas'
]].copy()

df_coca_silver = df_coca_silver.astype({
    'cod_dpto': str,
    'cod_mun': str,
    'ano': int,
    'hectareas': float
})

show_info(df_coca_silver, 'Coca - SILVER')
export_silver(df_coca_silver, 'coca')

# GOLD (renombrar columna para consistencia)
df_coca_gold = df_coca_silver.copy()
df_coca_gold = df_coca_gold.rename(columns={'hectareas': 'hectareas_coca'})

show_info(df_coca_gold, 'Coca - GOLD')
export_gold(df_coca_gold, 'coca')


COCA - SILVER
Shape: 7,337 filas x 6 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'hectareas']
Tipos:
cod_dpto        object
dpto_nombre     object
cod_mun         object
mun_nombre      object
ano              int64
hectareas      float64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,hectareas
0,91,AMAZONAS,91263,EL ENCANTO (Cor. Departamental),2001,191.8
1,91,AMAZONAS,91405,LA CHORRERA (Cor. Departamental),2001,65.0
2,91,AMAZONAS,91407,LA PEDRERA (Cor. Departamental),2001,0.0


  [SILVER] generated_data/silver/coca_silver.csv

COCA - GOLD
Shape: 7,337 filas x 6 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'hectareas_coca']
Tipos:
cod_dpto           object
dpto_nombre        object
cod_mun            object
mun_nombre         object
ano                 int64
hectareas_coca    float64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,hectareas_coca
0,91,AMAZONAS,91263,EL ENCANTO (Cor. Departamental),2001,191.8
1,91,AMAZONAS,91405,LA CHORRERA (Cor. Departamental),2001,65.0
2,91,AMAZONAS,91407,LA PEDRERA (Cor. Departamental),2001,0.0


  [GOLD]   generated_data/gold/coca_gold.csv


### 2.7 Cobertura Movil

**Nota**: Dataset solo tiene informacion anual, no mensual.

In [9]:
# Cargar datos
df_cobertura_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/9mey-c8s8.json',
    dataset_name='Cobertura Movil'
)

# Limpieza
df_cobertura = df_cobertura_raw.copy()

df_cobertura = df_cobertura.rename(columns={
    'a_o': 'ano',
    'cod_departamento': 'cod_dpto',
    'departamento': 'dpto_nombre',
    'cod_municipio': 'cod_mun',
    'municipio': 'mun_nombre'
})

df_cobertura['cod_dpto'] = df_cobertura['cod_dpto'].astype(str).str.zfill(2)
df_cobertura['cod_mun'] = df_cobertura['cod_mun'].astype(str).str.zfill(5)
df_cobertura['ano'] = pd.to_numeric(df_cobertura['ano'], errors='coerce').astype('Int64')

# Convertir coberturas S/N a 1/0
cobertura_cols = ['cobertura_2g', 'cobertura_3g', 'cobertuta_4g', 'cobertura_lte', 'cobertura_5g']
for col in cobertura_cols:
    if col in df_cobertura.columns:
        df_cobertura[col] = df_cobertura[col].map({'S': 1, 'N': 0}).fillna(0).astype(int)

if 'cobertuta_4g' in df_cobertura.columns:
    df_cobertura = df_cobertura.rename(columns={'cobertuta_4g': 'cobertura_4g'})

# SILVER (todas las columnas originales)
df_cobertura_silver = df_cobertura.copy()

show_info(df_cobertura_silver, 'Cobertura Movil - SILVER')
export_silver(df_cobertura_silver, 'cobertura_movil')

# GOLD (agregado por municipio-ano)
agg_dict = {col: 'max' for col in ['cobertura_2g', 'cobertura_3g', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g'] 
            if col in df_cobertura_silver.columns}

df_cobertura_gold = (df_cobertura_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano'])
    .agg(agg_dict)
    .reset_index()
)

show_info(df_cobertura_gold, 'Cobertura Movil - GOLD')
export_gold(df_cobertura_gold, 'cobertura_movil')

Cargando 'Cobertura Movil'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 350,000 registros
  ... 400,000 registros
  ... 407,281 registros
  [OK] Total: 407,281 filas x 16 columnas

COBERTURA MOVIL - SILVER
Shape: 407,281 filas x 16 columnas
Columnas: ['ano', 'trimestre', 'proveedor', 'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'cabecera_municipal', 'cod_centro_poblado', 'centro_poblado', 'cobertura_2g', 'cobertura_3g', 'cobertura_hspa_hspa_dc', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g']
Tipos:
ano                        Int64
trimestre                 object
proveedor                 object
cod_dpto                  object
dpto_nombre               object
cod_mun                   object
mun_nombre                object
cabecera_municipal        object
cod_centro_poblado        object
centro_poblado            object
cobertura_2g               int64
cobertura_3g     

,ano,trimestre,proveedor,cod_dpto,dpto_nombre,cod_mun,mun_nombre,cabecera_municipal,cod_centro_poblado,centro_poblado,cobertura_2g,cobertura_3g,cobertura_hspa_hspa_dc,cobertura_4g,cobertura_lte,cobertura_5g
0,2023,3,COLOMBIA MOVIL S.A ESP,27,CHOCÓ,27250,EL LITORAL DEL SAN JUAN,N,27250034,TORDÓ,0,0,N,1,0,0
1,2023,3,COLOMBIA TELECOMUNICACIONES S.A. E.S.P.,05,ANTIOQUIA,05495,NECHÍ,N,05495003,LA CONCHA,1,1,S,1,0,0
2,2022,3,COLOMBIA MOVIL S.A ESP,70,SUCRE,70508,OVEJAS,N,70508006,DON GABRIEL,0,0,N,1,0,0


  [SILVER] generated_data/silver/cobertura_movil_silver.csv

COBERTURA MOVIL - GOLD
Shape: 10,095 filas x 10 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'cobertura_2g', 'cobertura_3g', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g']
Tipos:
cod_dpto         object
dpto_nombre      object
cod_mun          object
mun_nombre       object
ano               Int64
cobertura_2g      int64
cobertura_3g      int64
cobertura_4g      int64
cobertura_lte     int64
cobertura_5g      int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,cobertura_2g,cobertura_3g,cobertura_4g,cobertura_lte,cobertura_5g
0,05,ANTIOQUIA,05001,MEDELLÍN,2015,1,1,0,1,0
1,05,ANTIOQUIA,05001,MEDELLÍN,2016,1,1,0,1,0
2,05,ANTIOQUIA,05001,MEDELLÍN,2017,1,1,0,1,0


  [GOLD]   generated_data/gold/cobertura_movil_gold.csv


### 2.8 PIB Departamental

**Nota**: Este dataset solo tiene informacion a nivel departamental y anual.

In [19]:
# Cargar datos
df_pib_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/kgyi-qc7j.json',
    dataset_name='PIB'
)

# Limpieza
df_pib = df_pib_raw.copy()

df_pib = df_pib.rename(columns={
    'a_o': 'ano',
    'c_digo_departamento_divipola': 'cod_dpto',
    'departamento': 'dpto_nombre',
    'valor_miles_de_millones_de': 'valor_agregado'
})

df_pib['cod_dpto'] = df_pib['cod_dpto'].astype(str).str.zfill(2)
df_pib['ano'] = pd.to_numeric(df_pib['ano'], errors='coerce').astype('Int64')
df_pib['valor_agregado'] = pd.to_numeric(df_pib['valor_agregado'], errors='coerce')

# SILVER (con actividad económica)
df_pib_silver = df_pib.copy()

show_info(df_pib_silver, 'PIB - SILVER')
export_silver(df_pib_silver, 'pib')

# GOLD (agregado total)
df_pib_gold = (df_pib_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'ano'])
    .agg(pib_total=('valor_agregado', 'sum'))
    .reset_index()
)

# Agregar población para PIB per capita
df_poblacion_for_pib = df_poblacion_gold[['cod_dpto', 'ano', 'poblacion_total']].copy()

df_pib_gold = df_pib_gold.merge(
    df_poblacion_for_pib,
    on=['cod_dpto', 'ano'],
    how='left'
)

df_pib_gold['pib_per_capita'] = (
    df_pib_gold['pib_total'] * 1_000_000_000 / df_pib_gold['poblacion_total']
).round(2)

df_pib_gold = df_pib_gold.astype({
    'cod_dpto': str,
    'ano': int
})

show_info(df_pib_gold, 'PIB - GOLD')
export_gold(df_pib_gold, 'pib_departamental')

Cargando 'PIB'...
  ... 16,302 registros
  [OK] Total: 16,302 filas x 7 columnas

PIB - SILVER
Shape: 16,302 filas x 7 columnas
Columnas: ['ano', 'actividad', 'sector', 'tipo_de_precios', 'cod_dpto', 'dpto_nombre', 'valor_agregado']
Tipos:
ano                  Int64
actividad           object
sector              object
tipo_de_precios     object
cod_dpto            object
dpto_nombre         object
valor_agregado     float64

Primeras filas:


,ano,actividad,sector,tipo_de_precios,cod_dpto,dpto_nombre,valor_agregado
0,2005,"Agricultura, ganadería, caza, silvicultura y p...",Primario,PIB a precios constantes de 2015,91,Amazonas,114.17
1,2005,"Agricultura, ganadería, caza, silvicultura y p...",Primario,PIB a precios constantes de 2015,05,Antioquia,4856.14
2,2005,"Agricultura, ganadería, caza, silvicultura y p...",Primario,PIB a precios constantes de 2015,81,Arauca,469.03


  [SILVER] generated_data/silver/pib_silver.csv

PIB - GOLD
Shape: 4,983 filas x 6 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'ano', 'pib_total', 'poblacion_total', 'pib_per_capita']
Tipos:
cod_dpto            object
dpto_nombre         object
ano                  int64
pib_total          float64
poblacion_total      Int64
pib_per_capita     Float64

Primeras filas:


,cod_dpto,dpto_nombre,ano,pib_total,poblacion_total,pib_per_capita
0,05,Antioquia,2005,126955.31,<NA>,<NA>
1,05,Antioquia,2006,138385.05,<NA>,<NA>
2,05,Antioquia,2007,151345.68,<NA>,<NA>


  [GOLD]   generated_data/gold/pib_departamental_gold.csv


### 2.9 Atentados Terroristas

**Nota**: Dataset solo tiene ano, no fecha detallada.

In [20]:
# Cargar datos
df_atentados_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/yfd7-8c9d.json',
    dataset_name='Atentados'
)

Cargando 'Atentados'...
  ... 282 registros
  [OK] Total: 282 filas x 27 columnas


In [22]:
# Limpieza
df_atentados = df_atentados_raw.copy()

# Renombrar columnas (usar nombres reales del API)
df_atentados = df_atentados.rename(columns={
    'a_o': 'ano',
    'c_digo_dane_de_municipio': 'cod_mun',
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre',
    'total_de_v_ctimas_del_caso': 'total_victimas',
    'total_civiles': 'total_civiles',
    'total_combatientes': 'total_combatientes',
    'lesionados_civiles': 'lesionados_civiles',
    'abandono_o_despojo_forzado': 'abandono_despojo',
    'amenaza_o_intimidaci_n': 'amenaza_intimidacion',
    'ataque_contra_misi_n_m_dica': 'ataque_mision_medica',
    'confinamiento_o_restricci': 'confinamiento',
    'desplazamiento_forzado': 'desplazamiento_forzado',
    'extorsi_n': 'extorsion_asociada',
    'pillaje': 'pillaje',
    'tortura': 'tortura'
})

# Estandarizar codigo municipio y extraer departamento
df_atentados['cod_mun'] = df_atentados['cod_mun'].astype(str).str.zfill(5)
df_atentados['cod_dpto'] = df_atentados['cod_mun'].str[:2]
df_atentados['ano'] = pd.to_numeric(df_atentados['ano'], errors='coerce').astype('Int64')
df_atentados['mes'] = pd.to_numeric(df_atentados['mes'], errors='coerce').astype('Int64')

# Convertir variables numericas
numeric_cols = [
    'total_victimas', 'total_civiles', 'total_combatientes', 'lesionados_civiles',
    'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica', 
    'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada', 
    'pillaje', 'tortura'
]
for col in numeric_cols:
    if col in df_atentados.columns:
        df_atentados[col] = pd.to_numeric(df_atentados[col], errors='coerce').fillna(0).astype(int)

# SILVER (todos los detalles)
df_atentados_silver = df_atentados[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes',
    'total_victimas', 'total_civiles', 'total_combatientes', 'lesionados_civiles',
    'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica',
    'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada',
    'pillaje', 'tortura'
]].copy()

show_info(df_atentados_silver, 'Atentados - SILVER')
export_silver(df_atentados_silver, 'atentados')

# GOLD (agregado)
df_atentados_gold = (df_atentados_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg({
        'total_victimas': 'sum',
        'total_civiles': 'sum',
        'total_combatientes': 'sum',
        'lesionados_civiles': 'sum',
        'abandono_despojo': 'sum',
        'amenaza_intimidacion': 'sum',
        'ataque_mision_medica': 'sum',
        'confinamiento': 'sum',
        'desplazamiento_forzado': 'sum',
        'extorsion_asociada': 'sum',
        'pillaje': 'sum',
        'tortura': 'sum'
    })
    .reset_index()
)

# Calcular total de atentados
df_atentados_gold['total_atentados'] = df_atentados_gold[[
    'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica',
    'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada',
    'pillaje', 'tortura'
]].sum(axis=1)

show_info(df_atentados_gold, 'Atentados - GOLD')
export_gold(df_atentados_gold, 'atentados')


ATENTADOS - SILVER
Shape: 282 filas x 18 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_victimas', 'total_civiles', 'total_combatientes', 'lesionados_civiles', 'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica', 'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada', 'pillaje', 'tortura']
Tipos:
cod_dpto                  object
dpto_nombre               object
cod_mun                   object
mun_nombre                object
ano                        Int64
mes                        Int64
total_victimas             int64
total_civiles              int64
total_combatientes         int64
lesionados_civiles         int64
abandono_despojo           int64
amenaza_intimidacion       int64
ataque_mision_medica       int64
confinamiento              int64
desplazamiento_forzado     int64
extorsion_asociada         int64
pillaje                    int64
tortura                    int64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_victimas,total_civiles,total_combatientes,lesionados_civiles,abandono_despojo,amenaza_intimidacion,ataque_mision_medica,confinamiento,desplazamiento_forzado,extorsion_asociada,pillaje,tortura
0,11,"BOGOTA, D. C.",11001,"BOGOTA, D.C.",1977,1,0,0,0,0,0,0,0,0,0,0,0,0
1,18,CAQUETA,18001,FLORENCIA,1978,8,0,0,0,0,0,0,0,0,0,0,0,0
2,76,VALLE DEL CAUCA,76001,SANTIAGO DE CALI,1982,8,0,0,0,6,0,0,0,0,0,0,0,0


  [SILVER] generated_data/silver/atentados_silver.csv

ATENTADOS - GOLD
Shape: 268 filas x 19 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_victimas', 'total_civiles', 'total_combatientes', 'lesionados_civiles', 'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica', 'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada', 'pillaje', 'tortura', 'total_atentados']
Tipos:
cod_dpto                  object
dpto_nombre               object
cod_mun                   object
mun_nombre                object
ano                        Int64
mes                        Int64
total_victimas             int64
total_civiles              int64
total_combatientes         int64
lesionados_civiles         int64
abandono_despojo           int64
amenaza_intimidacion       int64
ataque_mision_medica       int64
confinamiento              int64
desplazamiento_forzado     int64
extorsion_asociada         int64
pillaje                    int64

,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_victimas,total_civiles,total_combatientes,lesionados_civiles,abandono_despojo,amenaza_intimidacion,ataque_mision_medica,confinamiento,desplazamiento_forzado,extorsion_asociada,pillaje,tortura,total_atentados
0,05,ANTIOQUIA,05001,MEDELLIN,1983,11,0,0,0,0,0,0,0,0,0,0,0,0,0
1,05,ANTIOQUIA,05001,MEDELLIN,1985,5,0,0,0,0,0,0,0,0,0,0,0,0,0
2,05,ANTIOQUIA,05001,MEDELLIN,1985,9,0,0,0,8,0,0,0,0,0,0,0,0,0


  [GOLD]   generated_data/gold/atentados_gold.csv


## 2.10 Estupefacientes

In [23]:
# Cargar datos
df_estupef_raw = fetch_api_data(
    api_url='https://www.datos.gov.co/resource/kk69-w2jj.json',
    dataset_name='Estupefacientes'
)

# Limpieza
df_estupefacientes = df_estupef_raw.copy()

Cargando 'Estupefacientes'...
  ... 50,000 registros
  ... 100,000 registros
  ... 150,000 registros
  ... 200,000 registros
  ... 250,000 registros
  ... 300,000 registros
  ... 350,000 registros
  ... 400,000 registros
  ... 450,000 registros
  ... 500,000 registros
  ... 550,000 registros
  ... 600,000 registros
  ... 650,000 registros
  ... 700,000 registros
  ... 750,000 registros
  ... 800,000 registros
  ... 850,000 registros
  ... 900,000 registros
  ... 950,000 registros
  ... 1,000,000 registros
  ... 1,050,000 registros
  ... 1,100,000 registros
  ... 1,150,000 registros
  ... 1,200,000 registros
  ... 1,250,000 registros
  ... 1,300,000 registros
  ... 1,350,000 registros
  ... 1,400,000 registros
  ... 1,450,000 registros
  ... 1,500,000 registros
  ... 1,550,000 registros
  ... 1,600,000 registros
  ... 1,650,000 registros
  ... 1,700,000 registros
  ... 1,750,000 registros
  ... 1,800,000 registros
  ... 1,850,000 registros
  ... 1,900,000 registros
  ... 1,928,185 regis

In [24]:
df_estupefacientes = df_estupefacientes.rename(columns={
    'departamento': 'dpto_nombre',
    'municipio': 'mun_nombre',
    'codigo_dane': 'cod_mun',
    'clase_bien': 'tipo_sustancia',
    'fecha_hecho': 'fecha',
    'cantidad': 'cantidad'
})

# Parsear fecha y extraer ano y mes
df_estupefacientes['fecha'] = pd.to_datetime(df_estupefacientes['fecha'], format='%d/%m/%Y', errors='coerce')
df_estupefacientes['ano'] = df_estupefacientes['fecha'].dt.year
df_estupefacientes['mes'] = df_estupefacientes['fecha'].dt.month

# Estandarizar codigo municipio y extraer departamento
df_estupefacientes['cod_mun'] = df_estupefacientes['cod_mun'].astype(str).str.zfill(8).str[:5]
df_estupefacientes['cod_dpto'] = df_estupefacientes['cod_mun'].str[:2]

# Convertir cantidad a numerico
df_estupefacientes['cantidad'] = pd.to_numeric(df_estupefacientes['cantidad'], errors='coerce').fillna(0)

# Normalizar nombres de sustancias
df_estupefacientes['tipo_sustancia'] = (df_estupefacientes['tipo_sustancia']
    .str.upper()
    .str.strip()
    .str.replace(' ', '_')
)

# SILVER (con tipo de sustancia)
df_estupef_silver = df_estupefacientes[[
    'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre',
    'fecha', 'ano', 'mes', 'tipo_sustancia', 'cantidad'
]].copy()

print(f"\nTipos de sustancias: {df_estupef_silver['tipo_sustancia'].nunique()}")
print(df_estupef_silver['tipo_sustancia'].value_counts().head())

show_info(df_estupef_silver, 'Estupefacientes - SILVER')
export_silver(df_estupef_silver, 'estupefacientes')

# GOLD (agregado total)
df_estupef_gold = (df_estupef_silver
    .groupby(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes'])
    .agg(
        incautaciones_total=('cantidad', 'count'),
        cantidad_total_kg=('cantidad', 'sum')
    )
    .reset_index()
)

df_estupef_gold = df_estupef_gold.astype({
    'cod_dpto': str,
    'cod_mun': str,
    'ano': int,
    'mes': int,
    'incautaciones_total': int
})

show_info(df_estupef_gold, 'Estupefacientes - GOLD')
export_gold(df_estupef_gold, 'estupefacientes')


Tipos de sustancias: 5
tipo_sustancia
MARIHUANA       954374
BASUCO          452675
BASE_DE_COCA    300064
COCAINA         206359
HEROINA          14713
Name: count, dtype: int64

ESTUPEFACIENTES - SILVER
Shape: 1,928,185 filas x 9 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'fecha', 'ano', 'mes', 'tipo_sustancia', 'cantidad']
Tipos:
cod_dpto                  object
dpto_nombre               object
cod_mun                   object
mun_nombre                object
fecha             datetime64[ns]
ano                        int32
mes                        int32
tipo_sustancia            object
cantidad                 float64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,fecha,ano,mes,tipo_sustancia,cantidad
0,13,BOLÍVAR,13001,Cartagena (CT),2010-01-01,2010,1,MARIHUANA,28.0
1,66,RISARALDA,66400,La Virginia,2010-01-01,2010,1,BASUCO,2.0
2,08,ATLÁNTICO,08001,Barranquilla (CT),2010-01-01,2010,1,BASE_DE_COCA,32.5


  [SILVER] generated_data/silver/estupefacientes_silver.csv

ESTUPEFACIENTES - GOLD
Shape: 106,385 filas x 8 columnas
Columnas: ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'incautaciones_total', 'cantidad_total_kg']
Tipos:
cod_dpto                object
dpto_nombre             object
cod_mun                 object
mun_nombre              object
ano                      int64
mes                      int64
incautaciones_total      int64
cantidad_total_kg      float64

Primeras filas:


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,incautaciones_total,cantidad_total_kg
0,05,ANTIOQUIA,05001,Medellín (CT),2010,1,844,729569.34
1,05,ANTIOQUIA,05001,Medellín (CT),2010,2,965,1201978.70
2,05,ANTIOQUIA,05001,Medellín (CT),2010,3,692,299361.20


  [GOLD]   generated_data/gold/estupefacientes_gold.csv


## 3. Consolidacion de Datos

Merge de todos los datasets usando:
- `cod_dpto`: Codigo departamento (2 digitos)
- `cod_mun`: Codigo municipio (5 digitos)
- `ano`: Ano
- `mes`: Mes (1-12)

**Nota**: Los datasets que solo tienen informacion anual (poblacion, coca, cobertura, PIB, atentados) se replican para cada mes.

In [26]:
# Lista de datasets mensuales (con mes)
datasets_mensuales = [
    ('extorsion', df_extorsion_gold, ['total_extorsion']),
    ('homicidio', df_homicidio_gold, ['total_homicidio']),
    ('hurto', df_hurto_gold, ['total_hurto']),
    ('terrorismo', df_terrorismo_gold, ['total_terrorismo']),
    ('estupefacientes', df_estupef_gold, ['incautaciones_total', 'cantidad_total_kg']),
]

# Lista de datasets anuales (sin mes)
datasets_anuales = [
    ('coca', df_coca_gold, ['hectareas_coca']),
    ('cobertura', df_cobertura_gold, ['cobertura_2g', 'cobertura_3g', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g']),
    ('atentados', df_atentados_gold, ['total_atentados']),
]

print("Datasets mensuales:")
for name, df, cols in datasets_mensuales:
    print(f"  - {name}: {len(df):,} filas, variables: {cols}")

print("\nDatasets anuales (se replicaran por mes):")
for name, df, cols in datasets_anuales:
    print(f"  - {name}: {len(df):,} filas, variables: {cols}")

Datasets mensuales:
  - extorsion: 38,339 filas, variables: ['total_extorsion']
  - homicidio: 88,462 filas, variables: ['total_homicidio']
  - hurto: 109,660 filas, variables: ['total_hurto']
  - terrorismo: 9,369 filas, variables: ['total_terrorismo']
  - estupefacientes: 106,385 filas, variables: ['incautaciones_total', 'cantidad_total_kg']

Datasets anuales (se replicaran por mes):
  - coca: 7,337 filas, variables: ['hectareas_coca']
  - cobertura: 10,095 filas, variables: ['cobertura_2g', 'cobertura_3g', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g']
  - atentados: 268 filas, variables: ['total_atentados']


In [27]:
# Crear base del consolidado usando extorsion como referencia
merge_keys_mensual = ['cod_dpto', 'cod_mun', 'ano', 'mes']
merge_keys_anual = ['cod_dpto', 'cod_mun', 'ano']

# Empezar con extorsion (incluye nombres y mes)
df_consolidado = df_extorsion_gold[['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_extorsion']].copy()

print(f"Base inicial (extorsion): {df_consolidado.shape}")

# Merge con datasets mensuales
for name, df, value_cols in datasets_mensuales[1:]:  # Skip extorsion
    df_consolidado = df_consolidado.merge(
        df[merge_keys_mensual + value_cols],
        on=merge_keys_mensual,
        how='left'
    )
    print(f"  + {name}: {df_consolidado.shape}")

# Merge con datasets anuales (replicar por mes)
for name, df, value_cols in datasets_anuales:
    df_consolidado = df_consolidado.merge(
        df[merge_keys_anual + value_cols],
        on=merge_keys_anual,
        how='left'
    )
    print(f"  + {name} (anual): {df_consolidado.shape}")

# Agregar poblacion si existe
try:
    if 'df_poblacion_gold' in globals() and df_poblacion_gold is not None:
        df_consolidado = df_consolidado.merge(
            df_poblacion_gold[['cod_dpto', 'cod_mun', 'ano', 'poblacion_total']],
            on=['cod_dpto', 'cod_mun', 'ano'],
            how='left'
        )
        print(f"  + poblacion: {df_consolidado.shape}")
    else:
        print("  ⚠ poblacion no disponible")
except:
    print("  ⚠ poblacion no disponible")

# Agregar PIB
df_consolidado = df_consolidado.merge(
    df_pib_gold[['cod_dpto', 'ano', 'pib_total']],
    on=['cod_dpto', 'ano'],
    how='left'
)
print(f"  + pib: {df_consolidado.shape}")

print(f"\nConsolidado final: {df_consolidado.shape}")

Base inicial (extorsion): (38339, 7)
  + homicidio: (38339, 8)
  + hurto: (38339, 9)
  + terrorismo: (38339, 10)
  + estupefacientes: (38344, 12)
  + coca (anual): (38344, 13)
  + cobertura (anual): (38344, 18)
  + atentados (anual): (38415, 19)
  + poblacion: (38415, 20)
  + pib: (659788, 21)

Consolidado final: (659788, 21)


In [28]:
# Ver estructura del consolidado
print("Columnas del consolidado:")
print(df_consolidado.columns.tolist())
print(f"\nRango de anos: {df_consolidado['ano'].min()} - {df_consolidado['ano'].max()}")
print(f"Meses unicos: {sorted(df_consolidado['mes'].unique())}")
print(f"Departamentos unicos: {df_consolidado['cod_dpto'].nunique()}")
print(f"Municipios unicos: {df_consolidado['cod_mun'].nunique()}")

df_consolidado.head(10)

Columnas del consolidado:
['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_extorsion', 'total_homicidio', 'total_hurto', 'total_terrorismo', 'incautaciones_total', 'cantidad_total_kg', 'hectareas_coca', 'cobertura_2g', 'cobertura_3g', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g', 'total_atentados', 'poblacion_total', 'pib_total']

Rango de anos: 2003 - 2025
Meses unicos: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Departamentos unicos: 33
Municipios unicos: 1088


,cod_dpto,dpto_nombre,cod_mun,mun_nombre,ano,mes,total_extorsion,total_homicidio,total_hurto,total_terrorismo,...,cantidad_total_kg,hectareas_coca,cobertura_2g,cobertura_3g,cobertura_4g,cobertura_lte,cobertura_5g,total_atentados,poblacion_total,pib_total
0,05,ANTIOQUIA,05001,MEDELLIN,2003,1,23,210.0,99.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
1,05,ANTIOQUIA,05001,MEDELLIN,2003,2,15,167.0,87.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
2,05,ANTIOQUIA,05001,MEDELLIN,2003,3,9,199.0,127.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
3,05,ANTIOQUIA,05001,MEDELLIN,2003,4,11,195.0,142.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
4,05,ANTIOQUIA,05001,MEDELLIN,2003,5,9,181.0,174.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
5,05,ANTIOQUIA,05001,MEDELLIN,2003,6,7,163.0,149.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
6,05,ANTIOQUIA,05001,MEDELLIN,2003,7,13,151.0,210.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
7,05,ANTIOQUIA,05001,MEDELLIN,2003,8,12,128.0,221.0,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
8,05,ANTIOQUIA,05001,MEDELLIN,2003,9,7,163.0,213.0,4.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN
9,05,ANTIOQUIA,05001,MEDELLIN,2003,10,11,126.0,238.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,<NA>,NaN


In [29]:
# Resumen de valores nulos
print("Valores nulos por columna:")
null_summary = df_consolidado.isnull().sum()
null_pct = (null_summary / len(df_consolidado) * 100).round(1)
pd.DataFrame({'nulos': null_summary, 'porcentaje': null_pct}).query('nulos > 0')

Valores nulos por columna:


,nulos,porcentaje
total_homicidio,301933,45.8
total_hurto,137929,20.9
total_terrorismo,619223,93.9
incautaciones_total,226836,34.4
cantidad_total_kg,226836,34.4
hectareas_coca,478137,72.5
cobertura_2g,16917,2.6
cobertura_3g,16917,2.6
cobertura_4g,16917,2.6
cobertura_lte,16917,2.6


In [32]:
# Rangos de anos por dataset
rangos = {
    'extorsion': (df_extorsion_gold['ano'].min(), df_extorsion_gold['ano'].max()),
    'homicidio': (df_homicidio_gold['ano'].min(), df_homicidio_gold['ano'].max()),
    'hurto': (df_hurto_gold['ano'].min(), df_hurto_gold['ano'].max()),
    'terrorismo': (df_terrorismo_gold['ano'].min(), df_terrorismo_gold['ano'].max()),
    'coca': (df_coca_gold['ano'].min(), df_coca_gold['ano'].max()),
    'cobertura': (df_cobertura_gold['ano'].min(), df_cobertura_gold['ano'].max()),
    'pib': (df_pib_gold['ano'].min(), df_pib_gold['ano'].max()),
    'atentados': (df_atentados_gold['ano'].min(), df_atentados_gold['ano'].max()),
    'estupefacientes': (df_estupef_gold['ano'].min(), df_estupef_gold['ano'].max()),
}

print("Rangos de anos por dataset:")
print("-" * 40)
for name, (min_ano, max_ano) in rangos.items():
    print(f"  {name:15} : {min_ano} - {max_ano}")

# Calcular interseccion
ano_min_comun = max(r[0] for r in rangos.values())
ano_max_comun = min(r[1] for r in rangos.values())
print("-" * 40)
print(f"Interseccion comun: {ano_min_comun} - {ano_max_comun}")

Rangos de anos por dataset:
----------------------------------------
  extorsion       : 2003 - 2025
  homicidio       : 2003 - 2025
  hurto           : 2003 - 2025
  terrorismo      : 2003 - 2025
  coca            : 2001 - 2023
  cobertura       : 2015 - 2023
  pib             : 2005 - 2023
  atentados       : 1944 - 2024
  estupefacientes : 2010 - 2025
----------------------------------------
Interseccion comun: 2015 - 2023


## 4. Exportacion Final

In [35]:
# Filtrar ultimos 10 anos
ano_max = df_consolidado['ano'].max()
ano_min = ano_max - 9  # 10 anos incluido el actual

print(f"Rango seleccionado: {ano_min} - {ano_max} (10 anos)")

df_consolidado_10y = df_consolidado[
    (df_consolidado['ano'] >= ano_min) & 
    (df_consolidado['ano'] <= ano_max)
].copy()

print(f"Consolidado filtrado: {df_consolidado_10y.shape[0]:,} filas")
print(f"Anos incluidos: {sorted(df_consolidado_10y['ano'].unique())}")

# Ver cobertura de datos por ano
print("\nRegistros por ano:")
print(df_consolidado_10y.groupby('ano').size().sort_index())

# Ver nulos por columna
print("\nPorcentaje de nulos por variable:")
null_pct = (df_consolidado_10y.isnull().sum() / len(df_consolidado_10y) * 100).round(1)
print(null_pct[null_pct > 0])

# Ver columnas
print("\nColumnas:")
print(df_consolidado_10y.columns)

# Exportar
df_consolidado_10y.to_csv(f"{BASE_PATH}/consolidado_ultimos_10_anos_mensual.csv", index=False)
print(f"\nExportado: {BASE_PATH}/consolidado_ultimos_10_anos_mensual.csv")

Rango seleccionado: 2016 - 2025 (10 anos)
Consolidado filtrado: 647,185 filas
Anos incluidos: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

Registros por ano:
ano
2016      1794
2017      1995
2018      2499
2019      2814
2020    148150
2021    149343
2022    164298
2023    170073
2024      3307
2025      2912
dtype: int64

Porcentaje de nulos por variable:
total_homicidio        46.0
total_hurto            20.7
total_terrorismo       93.9
incautaciones_total    34.1
cantidad_total_kg      34.1
hectareas_coca         72.5
cobertura_2g            1.0
cobertura_3g            1.0
cobertura_4g            1.0
cobertura_lte           1.0
cobertura_5g            1.0
total_atentados        99.4
poblacion_total         1.4
pib_total               1.0
dtype: float64

Columnas:
Index(['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes',
       'total_extorsion', 'total

## 5. Resumen Final

In [37]:
print("="*70)
print("RESUMEN DE PROCESAMIENTO")
print("="*70)

print("\nArchivos generados:")
for root, dirs, files in os.walk(BASE_PATH):
    level = root.replace(BASE_PATH, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        filepath = os.path.join(root, file)
        size_mb = os.path.getsize(filepath) / (1024*1024)
        print(f"{subindent}{file} ({size_mb:.2f} MB)")

print(f"\nConsolidado final:")
print(f"   - Filas: {df_consolidado.shape[0]:,}")
print(f"   - Columnas: {df_consolidado.shape[1]}")
print(f"   - Anos: {int(df_consolidado['ano'].min())} - {int(df_consolidado['ano'].max())}")
print(f"   - Meses: 1 - 12")
print(f"   - Departamentos: {df_consolidado['cod_dpto'].nunique()}")
print(f"   - Municipios: {df_consolidado['cod_mun'].nunique()}")

print("\nProcesamiento completado.")

RESUMEN DE PROCESAMIENTO

Archivos generados:
generated_data/
  consolidado_ultimos_10_anos_mensual.csv (58.25 MB)
  DATOS.xlsx (0.33 MB)
  DATOS_consolidados.csv (0.81 MB)
  datos_preprocesados.csv (0.39 MB)
  .ipynb_checkpoints/
  clean/
    atentados_clean.csv (0.02 MB)
    cobertura_movil_clean.csv (0.43 MB)
    coca_clean.csv (0.29 MB)
    estupefacientes_clean.csv (4.61 MB)
    extorsion_clean.csv (1.41 MB)
    homicidio_clean.csv (3.28 MB)
    hurto_clean.csv (4.07 MB)
    pib_departamental_clean.csv (0.02 MB)
    poblacion_clean.csv (0.88 MB)
    terrorismo_clean.csv (0.34 MB)
  gold/
    atentados_gold.csv (0.02 MB)
    cobertura_movil_gold.csv (0.43 MB)
    coca_gold.csv (0.29 MB)
    estupefacientes_gold.csv (4.62 MB)
    extorsion_gold.csv (1.41 MB)
    homicidio_gold.csv (3.28 MB)
    hurto_gold.csv (4.07 MB)
    pib_departamental_gold.csv (0.22 MB)
    poblacion_gold.csv (0.88 MB)
    terrorismo_gold.csv (0.34 MB)
  model_outputs/
    01_valores_faltantes.png (0.19 MB)
  

## 3. Resumen de Datasets SILVER

Columnas disponibles en cada archivo silver (sin agregar):

In [38]:
import pandas as pd
from pathlib import Path

print("="*70)
print("COLUMNAS DE DATASETS SILVER (sin agregar)")
print("="*70)

silver_files = sorted(Path(SILVER_PATH).glob('*_silver.csv'))

for file_path in silver_files:
    dataset_name = file_path.stem.replace('_silver', '').upper()
    
    try:
        df = pd.read_csv(file_path, nrows=5)
        
        print(f"\n{dataset_name}")
        print(f"  Archivo: {file_path.name}")
        print(f"  Shape: {len(pd.read_csv(file_path)):,} filas")
        print(f"  Columnas ({len(df.columns)}): {list(df.columns)}")
        
        # Valores únicos de columnas categóricas
        for col in df.columns:
            if df[col].dtype == 'object' and col not in ['cod_dpto', 'cod_mun', 'dpto_nombre', 'mun_nombre', 'fecha_hecho', 'fecha']:
                unique_count = pd.read_csv(file_path)[col].nunique()
                if unique_count < 30:
                    unique_vals = pd.read_csv(file_path)[col].unique()[:10]
                    print(f"  ⚠ '{col}' ({unique_count} únicos): {list(unique_vals)}")
        
    except Exception as e:
        print(f"  ❌ Error: {e}")

print(f"\n{'='*70}")
print(f"Total archivos SILVER: {len(silver_files)}")
print(f"{'='*70}")

COLUMNAS DE DATASETS SILVER (sin agregar)

ATENTADOS
  Archivo: atentados_silver.csv
  Shape: 282 filas
  Columnas (18): ['cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'ano', 'mes', 'total_victimas', 'total_civiles', 'total_combatientes', 'lesionados_civiles', 'abandono_despojo', 'amenaza_intimidacion', 'ataque_mision_medica', 'confinamiento', 'desplazamiento_forzado', 'extorsion_asociada', 'pillaje', 'tortura']

COBERTURA_MOVIL
  Archivo: cobertura_movil_silver.csv
  Shape: 407,281 filas
  Columnas (16): ['ano', 'trimestre', 'proveedor', 'cod_dpto', 'dpto_nombre', 'cod_mun', 'mun_nombre', 'cabecera_municipal', 'cod_centro_poblado', 'centro_poblado', 'cobertura_2g', 'cobertura_3g', 'cobertura_hspa_hspa_dc', 'cobertura_4g', 'cobertura_lte', 'cobertura_5g']
  ⚠ 'proveedor' (8 únicos): ['COLOMBIA MOVIL  S.A ESP', 'COLOMBIA TELECOMUNICACIONES S.A. E.S.P.', 'AVANTEL S.A.S', 'COMUNICACION CELULAR S A COMCEL S A', 'EMPRESA DE TELECOMUNICACIONES DE BOGOTA S.A. ESP', 'PARTNERS TELECOM COL